# 📸 Image Captioning Part B: Training, CIDEr & System Design

Covers the full lifecycle: how models are trained, how CIDEr evaluates quality, and how a production captioning system is architected.

## Two-Stage Training

### Stage 1 — Pretrain Text Decoder on Text Corpora
- Train a language model (GPT-style) on large text datasets
- Objective: **next-token prediction** (autoregressive cross-entropy loss)
- Gives the decoder strong language priors before it ever sees an image

### Stage 2 — Finetune on Image-Caption Pairs
- Freeze (or lightly tune) the image encoder; train the decoder cross-attention layers on `(image, caption)` pairs
- The input sequence is: `[IMG_1] [IMG_2] ... [IMG_N] <BOS> w_1 w_2 ... w_T`
- Loss is computed **only on caption tokens** (not on image tokens)
- Teacher forcing: ground-truth tokens are fed as inputs; predicted logits are compared against the next ground-truth token

```
Loss = -1/T * sum_t [ log P(w_t | IMG_tokens, w_1 ... w_{t-1}) ]
```

Keeps compute efficient by reusing the pretrained vision encoder (e.g., ViT) and text decoder (e.g., GPT-2).

In [ ]:
import torch
import torch.nn.functional as F

# Simulated training step: image tokens + caption tokens
vocab_size = 50   # tiny vocab for demo
seq_len    = 6    # 3 image token slots + 3 caption tokens
n_img_tok  = 3    # image tokens occupy first 3 positions

torch.manual_seed(0)

# logits: model output for each position (batch_size=1)
logits = torch.randn(1, seq_len, vocab_size)  # (B, T, V)

# ground-truth target tokens (image tokens are -100 = ignored)
targets = torch.tensor([[-100, -100, -100, 12, 7, 34]])  # (B, T)

# Reshape for cross-entropy: (B*T, V) vs (B*T,)
loss = F.cross_entropy(
    logits.view(-1, vocab_size),
    targets.view(-1),
    ignore_index=-100   # skip image token positions
)

print(f"Caption token positions: {list(range(n_img_tok, seq_len))}")
print(f"Target caption tokens  : {targets[0, n_img_tok:].tolist()}")
print(f"Training loss (CE)     : {loss.item():.4f}")

## CIDEr Metric

**Consensus-based Image Description Evaluation (CIDEr)** measures how similar a generated caption is to a set of human reference captions, weighted by how informative each n-gram is.

### How It Works

1. **Extract n-grams** (typically n = 1..4) from candidate and all references
2. **TF-IDF weighting** — n-grams rare across the reference set get higher weight (IDF); n-grams frequent *within* a caption get higher weight (TF)
   - `TF(g, s)  = count(g in s) / total n-grams in s`
   - `IDF(g)    = log( |D| / df(g) )` where `|D|` = total captions in dataset
3. **Cosine similarity** between the TF-IDF vector of the candidate and each reference
4. **Average** across all references and n-gram orders; scale by 10

### Why CIDEr over BLEU?
- BLEU gives equal weight to all n-grams; CIDEr **down-weights common, uninformative words** ("a", "the", "is")
- CIDEr correlates better with human judgements on image captioning benchmarks (COCO)

In [ ]:
from collections import Counter
import math

def get_ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def cider_n(candidate, references, n=1):
    """Simplified CIDEr for a single n-gram order."""
    # TF for candidate
    cand_ng  = get_ngrams(candidate.split(), n)
    cand_len = max(sum(cand_ng.values()), 1)
    tf_cand  = {g: c / cand_len for g, c in cand_ng.items()}

    # IDF: count how many references contain each n-gram
    all_ref_ng = [get_ngrams(r.split(), n) for r in references]
    doc_freq   = Counter(g for ref_ng in all_ref_ng for g in ref_ng)
    N          = len(references)
    idf        = {g: math.log((N + 1) / (df + 1)) for g, df in doc_freq.items()}

    scores = []
    for ref_ng in all_ref_ng:
        ref_len = max(sum(ref_ng.values()), 1)
        tf_ref  = {g: c / ref_len for g, c in ref_ng.items()}
        all_g   = set(tf_cand) | set(tf_ref)
        dot  = sum(tf_cand.get(g,0) * idf.get(g,0) * tf_ref.get(g,0) * idf.get(g,0) for g in all_g)
        norm_c = math.sqrt(sum((tf_cand.get(g,0)*idf.get(g,0))**2 for g in tf_cand))
        norm_r = math.sqrt(sum((tf_ref.get(g,0) *idf.get(g,0))**2 for g in tf_ref))
        scores.append(dot / (norm_c * norm_r + 1e-9))
    return sum(scores) / len(scores)

references = [
    "a dog playing fetch in a sunny park",
    "a brown dog chasing a ball in the park",
    "a dog running after a ball outdoors",
]
candidate_good = "a dog playing with a ball in the park"
candidate_bad  = "a cat sitting on a windowsill"

for n in [1, 2]:
    print(f"CIDEr-{n}  good caption: {cider_n(candidate_good, references, n):.3f}")
    print(f"CIDEr-{n}  bad  caption: {cider_n(candidate_bad,  references, n):.3f}")

## System Design — End-to-End Captioning Pipeline

```
Raw Image
   |
   v
+------------------+
| Image Preprocess |  Resize to 224x224, normalize, optional augmentation
+------------------+
   |
   v
+------------------+
| Vision Encoder   |  ViT / CNN  ->  patch embeddings  (N x D)
+------------------+
   |
   v
+------------------+
| Projection Layer |  Linear / MLP aligns visual dim to text dim
+------------------+
   |
   v
+---------------------------+
| Text Decoder (autoregress)|  Beam search (k=5), max_len=30
| Cross-attention to image  |  Attends to visual tokens each step
+---------------------------+
   |
   v
+------------------+
| Post-processing  |  Detokenize, capitalize, remove repetitions
+------------------+
   |
   v
Final Caption (string)
```

**Key design choices**
- **Beam search** (not greedy) trades compute for higher-quality captions
- **Projection layer** avoids retraining the full encoder for each new backbone
- **Repetition penalty** in post-processing reduces degenerate outputs
- **Caching** visual embeddings at inference saves repeated encoder forward passes

## Interview Cheat Sheet

**Q1. ViT vs CNN as image encoder — which is better for captioning?**
> ViT tends to win on large datasets because self-attention can model long-range patch relationships (e.g., relating subject and background). CNNs are more data-efficient on smaller datasets. In practice, CLIP-ViT is the dominant choice in modern captioners (BLIP, LLaVA).

**Q2. Why beam search instead of greedy decoding?**
> Greedy picks the highest-probability token at each step, which can lead to locally optimal but globally poor sequences. Beam search keeps k hypotheses alive, exploring more of the probability space and typically producing more fluent and accurate captions.

**Q3. CIDEr vs BLEU — when does it matter?**
> CIDEr applies TF-IDF weighting so it rewards n-grams that are distinctive to the image ("golden retriever") over generic words ("the", "a"). It correlates much better with human judgements on COCO. Use BLEU when you need fast, cheap evaluation or comparison to legacy baselines.

**Q4. What is the role of cross-attention in the decoder?**
> Cross-attention allows each decoder token to query the visual feature map. The decoder forms a query from its current hidden state, and keys/values come from image patch embeddings. This lets the model selectively focus on relevant image regions when generating each word.

**Q5. How do you prevent hallucination in image captioning?**
> (a) Use contrastive decoding or nucleus sampling to avoid over-confident token choices. (b) Penalize tokens not grounded in high-attention image regions. (c) RLHF / RLAIF with a reward model that penalizes factually incorrect captions. (d) Post-hoc verification with a visual QA model.

## Quiz

**Q1. During training, why is the cross-entropy loss masked to ignore image token positions?**

<details>
<summary>Show answer</summary>

Image tokens are continuous embeddings projected from the vision encoder — they have no ground-truth discrete token ID to predict against. Including them in the loss would be undefined (or require a separate reconstruction objective). Masking them with `ignore_index=-100` ensures the model is only trained to predict caption words.

</details>

---

**Q2. A CIDEr score of 0 does not necessarily mean the caption is wrong. Why?**

<details>
<summary>Show answer</summary>

CIDEr measures n-gram overlap with a specific set of reference captions. If the candidate uses different but equally valid phrasing (synonyms, different sentence structure), the n-gram overlap may be zero even though the description is semantically correct. This is a known limitation of all n-gram-based metrics.

</details>

---

**Q3. In beam search with beam width k=5, what is the space complexity relative to greedy decoding?**

<details>
<summary>Show answer</summary>

Space complexity scales as O(k * T * D) where T is max sequence length and D is hidden dimension, because you must maintain k separate KV-cache states in parallel. Greedy is O(T * D). For k=5 this is a 5x memory overhead, which is why very large beam widths are rarely used in production.

</details>